---
title: Parts-Based Decomposition via Disjoint Basis Learning
bibliography: [dictionary-learning.bib, nmf.bib, Pattern_Recognition_and_Machine_Learning.bibtex, gaussian-scaled-mixture.bib,olivetti-faces.bib]
---

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

Lately I've been working on neural network interpretability, looking inside a network to understand why it behaves the way it does. A core challenge is decomposing internal representations into meaningful parts. The choice of *how* you decompose matters quite a bit — different assumptions lead to very different results.

Many algorithms exist for this, each with their own take:

- **Dictionary learning** [@OLSHAUSEN19973311]: Reconstructs data using a *sparse* combination of basis vectors. Widely used in interpretability.
- **Non-negative Matrix Factorization (NMF)** [@Lee1999]: Assumes non-negative inputs and coefficients, which naturally leads to parts-based representations.
- **Independent Component Analysis (ICA)** (TODO: CITE): Finds basis vectors that are statistically independent and non-Gaussian.

What I really want is something like NMF, a parts-based decomposition. The problem is that neural network activations can be negative, and NMF doesn't handle that. In this article, I'm going to write about a method which constrains the basis vectors in a way that leads to parts-based representation using gradient descent.  
Like NMF, the disjoint support constraint needs to be soft, we do this by adding a penalty to the basis vectors.   

::: {.callout-warning}
The parts-based representation here has no spatial notion, dimensions that are close to each other do not cluster.
:::


# Problem statement and notation


We are given a dataset with $N$ samples, each with $C$ dimensions, represented as a matrix $X \in \mathbb{R}^{N \times C}$, where each row is a sample.

We approximate $X$ as:
$$\hat{X} = SW$$

where:

- $S \in \mathbb{R}^{N \times R}$ is the coefficient matrix, with row $i$ denoted $\vec{s_{r_i}}$
- $W \in \mathbb{R}^{R \times C}$ is the basis matrix, where:
  - Row $i$ is denoted $\vec{w_{r_i}}$
  - Column $j$ is denoted $\vec{w_{c_j}}$
  - The scalar at row $i$, column $j$ is $w_{r_i c_j}$

::: {.callout-note}
We will be working with both rows and columns in the $W$ matrix. This is why I'm moving away from the standard notation for the extra expressivity.   
:::

Additionally, we constrain $W$ such that for any column $j$, at most one row of $W$ has a non-zero entry. Equivalently, the rows of $W$ have **disjoint support**, meaning no two basis vectors are active on the same dimension.

# Comparison with existing algorithms

The most relevant existing algorithms are ICA and NMF (as mentioned above).  
I'll be comparing the results of the algorithm here with ICA and NMF on two datasets:

- Olivetti faces dataset [@341300]
- A synthetic dataset created by combining ground truth disjoint basis vectors. This is closer to what we see in interpretability.  


## Olivetti faces

I'll be using the [scikit-learn decomposition guide on the Olivetti faces dataset](https://scikit-learn.org/stable/auto_examples/decomposition/plot_faces_decomposition.html) as a reference. The code is taken as-is from that page for the NMF and ICA results.

Along with the basis vectors, I also show the Gram matrix of the basis vectors. The off-diagonal elements for our algorithm are closer to 0 compared to ICA and NMF, simply because we explicitly penalize the model on this constraint.

The Olivetti faces dataset is not the best fit for our algorithm. Our method works well when $X$ can be reconstructed using a simple linear combination of parts — this assumption doesn't hold well for complex datasets like Olivetti. Without strong theoretical guidance on hyperparameter selection, getting good results here requires a lot of trial and error.

I'm including it anyway as it is a standard benchmark for parts-based decomposition methods.

In [ ]:
# | code-fold: true
# | code-summary: helpers for downloading the dataset and drawing


import torch
import logging
from pt_to_api.utils import show_gram

import matplotlib.pyplot as plt
from numpy.random import RandomState

from sklearn import cluster, decomposition
from sklearn.datasets import fetch_olivetti_faces

rng = RandomState(0)

# Display progress logs on stdout
logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s")

faces, _ = fetch_olivetti_faces(return_X_y=True, shuffle=True, random_state=rng)
n_samples, n_features = faces.shape

# Global centering (focus on one feature, centering all samples)
faces_centered = faces - faces.mean(axis=0)

# Local centering (focus on one sample, centering all features)
faces_centered -= faces_centered.mean(axis=1).reshape(n_samples, -1)


n_row, n_col = 2, 5
n_components = n_row * n_col
image_shape = (64, 64)


def plot_gallery(title, images, n_col=n_col, n_row=n_row, cmap=plt.cm.gray):
    fig, axs = plt.subplots(
        nrows=n_row,
        ncols=n_col,
        figsize=(2.0 * n_col, 2.3 * n_row),
        facecolor="white",
        constrained_layout=True,
    )
    fig.get_layout_engine().set(w_pad=0.01, h_pad=0.02, hspace=0, wspace=0)
    fig.set_edgecolor("black")
    fig.suptitle(title, size=16)
    for ax, vec in zip(axs.flat, images):
        vmax = max(vec.max(), -vec.min())
        im = ax.imshow(
            vec.reshape(image_shape),
            cmap=cmap,
            interpolation="nearest",
            vmin=-vmax,
            vmax=vmax,
        )
        ax.axis("off")

    fig.colorbar(im, ax=axs, orientation="horizontal", shrink=0.99, aspect=40, pad=0.01)
    plt.show()


def plot_gallery_with_gram(title, images, W, n_col=n_col, n_row=n_row, cmap=plt.cm.gray):
    if not isinstance(W, torch.Tensor):
        W = torch.tensor(W)
    W_norm = W / (W.norm(dim=0, keepdim=True) + 1e-8)
    gram = (W_norm.T @ W_norm).numpy()

    fig = plt.figure(
        figsize=(2.0 * (n_col + 1), 2.3 * n_row),
        facecolor="white",
        constrained_layout=True,
    )
    fig.get_layout_engine().set(w_pad=0.01, h_pad=0.02, hspace=0, wspace=0)
    fig.set_edgecolor("black")
    fig.suptitle(title, size=16)

    gs = fig.add_gridspec(nrows=n_row, ncols=n_col + 1)

    # Gallery axes
    gallery_axs = [fig.add_subplot(gs[r, c]) for r in range(n_row) for c in range(n_col)]
    for ax, vec in zip(gallery_axs, images):
        vmax = max(vec.max(), -vec.min())
        im = ax.imshow(
            vec.reshape(image_shape),
            cmap=cmap,
            interpolation="nearest",
            vmin=-vmax,
            vmax=vmax,
        )
        ax.axis("off")
    for ax in gallery_axs[len(images):]:
        ax.axis("off")

    # Gram axis spans all rows in the last column
    gram_ax = fig.add_subplot(gs[:, -1])
    gram_ax.imshow(gram, cmap="gray", interpolation="nearest")
    gram_ax.set_title("Gram", fontsize=10)
    # gram_ax.axis("off")

    fig.colorbar(im, ax=gallery_axs, orientation="horizontal",
                 shrink=0.99, aspect=40, pad=0.01)
    plt.show()

plot_gallery("Sample faces from the dataset", faces_centered[:n_components])

In [ ]:
faces.std()

In [ ]:
# | code-fold: true
nmf_estimator = decomposition.NMF(n_components=n_components, tol=5e-3)
nmf_estimator.fit(faces)  # original non- negative dataset
plot_gallery_with_gram(
    "Non-negative components - NMF", nmf_estimator.components_, nmf_estimator.components_.T
)

In [ ]:
# | code-fold: true
ica_estimator = decomposition.FastICA(
    n_components=n_components, max_iter=10000, whiten="arbitrary-variance", tol=15e-5
)
ica_estimator.fit(faces_centered)
plot_gallery_with_gram(
    "Independent components - FastICA", ica_estimator.components_, ica_estimator.components_.T
)

In [ ]:
# | code-fold: true
from pt_to_api import disjoint_ae

std_eps = 1e-3
sigma_0 = std_eps * 10
sigma_s = sigma_0 * 3
alpha = 5000 / (sigma_0*sigma_0)


model, codes, components, recon = disjoint_ae.train(
    faces, n_components, lr=1e-3, epochs=3000, verbose=False, sigma_0=sigma_0, sigma_s=sigma_s, alpha=alpha, sigma_eps=std_eps,
)

plot_gallery_with_gram(
    "Disjoint Basis Finder, on non-negative data (same format used in NMF)", components, components.T
)

In [ ]:
# | code-fold: true
std_eps = 1e-3
sigma_0 = std_eps * 10
sigma_s = sigma_0 * 5
alpha = 5000 / (sigma_0*sigma_0)


model, codes, components, recon = disjoint_ae.train(
    faces_centered, n_components, lr=1e-3, epochs=1000, verbose=False, sigma_0=sigma_0, sigma_s=sigma_s, alpha=alpha, sigma_eps=std_eps,
)

plot_gallery_with_gram(
    "Disjoint Basis Finder, on whitenned data (same format used in ICA)", components, components.T
)

## Synthetic dataset

We create $K$ ground truth vectors, $\vec{w_k}$, along with random $\vec{S}$ sampled from a Gaussian. $\vec{x_i} = \sum_{k=0}^{K} (s_{ik}\vec{w_k}) + \epsilon$. $\epsilon$ is a small random noise, also sampled from Gaussian.  
The ground truth vectors have disjoint support. $\vec{X}$ is generally very clean for now. We create $N$ copies of $\vec{R}$ for generating the dataset $X$


::: {.callout-note}
The dataset generation code creates very clean datasets right now (there is no overlap between disjoint atoms, very less noise). It's not very representative of the real world, but it's a good baseline for now
:::

In [ ]:
# | code-fold: true
# | code-summary: Open helper function definitions

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import matplotlib.pyplot as plt
from pt_to_api.utils import show_single_channel_red_green_black as S

MODE = "light"


def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]


def generate_synthetic_patches(
    patch_dim=72, n_components=10, k=3, n_samples=1000, noise_std=0.01, seed=42
):
    rng = np.random.RandomState(seed)

    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    # each sample uses exactly k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        idx = rng.choice(n_components, k, replace=False)
        codes_true[i, idx] = rng.randn(k)

    X = codes_true @ W_true
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition


def show_closest_component_of_W_for_each_component(components, W_true, figsize=(5, 2)):
    """
    Given two arrays of numpy vectors of same shapes
    for every component in `components`, this function shows the array in `W_true`
    which has the maximum cosine similarity with the component
    """
    sims = np.abs(cosine_similarity(components, W_true))
    pairs = []
    for i in range(len(components)):
        j = np.argmax(sims[i])
        pairs.append((i, j, sims[i][j]))
    for i, j, score in pairs:
        S(
            [components[i].reshape(3, 3), W_true[j].reshape(3, 3)],
            figsize,
            mode=MODE,
            suptitle=f"similarity score={score}",
            ax_titles=["component", "ground_truth"],
        )
        plt.show()

In [ ]:
# | code-fold: true


X, W_true, codes_true, dim_partition = generate_synthetic_patches(
    patch_dim=9, n_components=3, k=3
)
ws_to_show = [w.reshape(3, 3) for w in W_true]
S(
    ws_to_show,
    (8, 2),
    ncols=3,
    mode=MODE,
    suptitle="ground truth basis vectors \n[bright green = high positive, bright red = high negative, white = near zero]\nA vector is of size 1x9, but is shown as 3x3 for easier visibility",
)
plt.show()
S(
    [X[0].reshape(3, 3)] + ws_to_show,
    (8, 2),
    4,
    suptitle="First input, with its basis components, the coefficient of each component is it's title",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[0][0]:.3f}",
        f"{codes_true[0][1]:.3f}",
        f"{codes_true[0][2]:.3f}",
    ],
)
plt.show()
S(
    [X[1].reshape(3, 3)] + ws_to_show,
    (8, 2),
    4,
    suptitle="Second input",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[1][0]:.3f}",
        f"{codes_true[1][1]:.3f}",
        f"{codes_true[1][2]:.3f}",
    ],
)
plt.show()

# Probabilistic model

Consider the random variables:

- $W$: the collection of basis vectors (matrix with $K$ components × $D$ dimensions)
- $\vec{s}$: a vector of coefficients for a single input example
- $\vec{x}$: a vector of a single input

We want to find $p(W\vec{s}|\vec{x})$. From Bayesian rule:

$$p(W\vec{s}|\vec{x}) = \frac{p(\vec{x}|W\vec{s})P(W|\vec{s})p(\vec{s})}{p(\vec{x})}$$

$p(\vec{x})$ is a marginalizing constant, and can be ignored.  

$$p(W\vec{s}|\vec{x}) \propto p(\vec{x}|W\vec{s})P(W|\vec{s})p(\vec{s})$$

We make the following modeling choices:

- $p(\vec{x}|W\vec{s}) = \mathcal{N}(\vec{x}|W\vec{s}, \sigma_{\epsilon}^2)$, modeling $\vec{x}$ as $W\vec{s} + \epsilon$ with Gaussian noise
- $p(W|\vec{s}) = p(W)$, assuming $W$ and $\vec{s}$ are sampled independently
- $p(\vec{s}) = \mathcal{N}(\vec{s}|0, \sigma_s^2)$, a Gaussian prior on $\vec{s}$ to regularize optimization; without it, gradient descent overfits $\vec{s}$ to the reconstruction

$$p(W\vec{s}|\vec{x}) \propto \mathcal{N}(\vec{x}|W\vec{s}, \sigma_{\epsilon}^2) \mathcal{N}(\vec{s}|0, \sigma_s^2) P(W)$$

The key modeling choice is the prior $P(W)$, which we constrain to enforce disjoint support.

## Constraining $p(W)$ for disjoint support

We denote the element at row $i$, column $j$ of $W$ as $w_{r_i c_j}$, where $r_i$ is the $i$-th component ($i \in [0, K-1]$) and $c_j$ is the $j$-th dimension ($j \in [0, D-1]$).


Writing $p(W)$ in terms of its components:  

$$p(W) = p(w_{r_0 c_0})p(w_{r_0 c_1}|w_{r_0 c_0})p(w_{r_0 c_2}|w_{r_0 c_0},w_{r_0 c_1}) \cdots p(w_{r_{K-1} c_{D-1}}|w_{r_0 c_0}, w_{r_0 c_1}, \ldots, w_{r_{K-1} c_{D-2}})$$

The goal is to ensure no two basis vectors share a non-zero value in the same column. The disjoint support constraint operates column-wise: within each column, at most one entry may be non-zero. Columns are independent of one another. We can simplify the equation with this assumption.  

Let $\vec{w}_{c_j} = [w_{r_0 c_j}, \ldots, w_{r_{K-1} c_j}]^T$ denote the vector of all component weights for dimension $c_j$. 

$$p(W) = \prod_{j=0}^{D-1} p(\vec{w}_{c_j}) $$


### Constraining Each Column

The constraint has been delegated to $p(\vec{w}_{c_j})$. Expanding by the product rule:

$$p(\vec{w}_{c_j}) = p(w_{r_0 c_j})p(w_{r_1 c_j}|w_{r_0 c_j})\cdots p(w_{r_{K-1} c_j}|w_{r_0 c_j}, \ldots, w_{r_{K-2} c_j})$$

The disjoint support constraint requires each conditional $p(w_{r_i c_j}|w_{r_0 c_j}, \ldots, w_{r_{i-1} c_j})$ to concentrate mass at zero if any previous entry in the column is non-zero, and sample freely otherwise. A natural formulation is a mixture:

- sample from $\mathcal{N}(0, \sigma_{w_{c_j}}^2)$ when no non-zero value has appeared in the column yet
- sample from $\text{Laplace}(0, b)$ otherwise, concentrating mass at zero

However, this leads to a combinatorial explosion — the number of cases doubles at each step, producing a tree of $2^K$ leaves.

Instead, we use Gaussian scale mixtures [@10.1214/06-BA117A]: a Gaussian with sufficiently small variance concentrates mass near zero, approximating a spike. This lets the variance itself carry the constraint:

$$p(w_{r_i c_j}|w_{r_0 c_j}, \ldots, w_{r_{i-1} c_j}) = \mathcal{N}(0, \sigma_{w_{r_i c_j}}^2)\, p(\sigma_{w_{r_i c_j}}^2|w_{r_0 c_j}, \ldots, w_{r_{i-1} c_j})$$

To simplify, we treat $\sigma_{w_{r_i c_j}}^2$ as a deterministic function of the previous entries rather than a random variable. This is achieved by setting:

$$p(\sigma_{w_{r_i c_j}}^2|w_{r_0 c_j}, \ldots, w_{r_{i-1} c_j}) = \delta\!\left(\sigma_{w_{r_i c_j}}^2 - f(w_{r_0 c_j}, \ldots, w_{r_{i-1} c_j})\right)$$

which collapses the distribution to a point mass at $\sigma_{w_{r_i c_j}}^2 = f(w_{r_0 c_j}, \ldots, w_{r_{i-1} c_j})$, giving:

$$p(w_{r_i c_j}|w_{r_0 c_j}, \ldots, w_{r_{i-1} c_j}) = \mathcal{N}(0, \sigma_{w_{r_i c_j}}^2) \quad \text{where} \quad \sigma_{w_{r_i c_j}}^2 = f(w_{r_0 c_j}, \ldots, w_{r_{i-1} c_j})$$

::: {.callout-caution}
Laplace distributions produce exact zeros under optimization. The Gaussian approximation used here instead concentrates mass near zero,

:::

### Finding $f$

In the base case, $p(w_{r_0 c_j}) = \mathcal{N}(0, \sigma_0^2)$. For $i > 0$, we require $\sigma_{w_{r_i c_j}}^2$ to collapse toward zero when any preceding entry in the column is non-zero. We measure this via $x = \sum_{k=0}^{i-1} w_{r_k c_j}^2$ and choose $f$ such that $\sigma^2$ decreases rapidly as $x$ increases. We consider two candidates:

- Laplace kernel: $\sigma_{w_{r_i c_j}}^2 = \sigma_0^2 e^{\frac{-|x|}{b}}$
- Lorentzian: $\sigma_{w_{r_i c_j}}^2 = \frac{\sigma_0^2}{1 + \alpha x^2}$

These are plotted for various hyperparameter choices below. 


In [ ]:
# | code-fold: true
import matplotlib.pyplot as plt

x = np.concatenate([np.linspace(-10, 0, 250), np.linspace(0, 10, 250)])

sigma0_sq = 1.0

b_values = [1, 0.5, 0.1, 0.01]
alpha_values = [1, 10, 100, 1000]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Laplace kernel
ax = axes[0]
for b in b_values:
    y = sigma0_sq * np.exp(-np.abs(x) / b)
    ax.plot(x, y, label=f"b = {b}")
ax.set_title("Laplace Kernel")
ax.set_xlabel("x")
ax.set_ylabel("σ²(x)")
ax.legend()
ax.grid(True, alpha=0.3)

# Lorentzian
ax = axes[1]
for alpha in alpha_values:
    y = sigma0_sq / (1 + alpha * x**2)
    ax.plot(x, y, label=f"α = {alpha}")
ax.set_title("Lorentzian")
ax.set_xlabel("x")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("kernels.png", dpi=150)
plt.show()

We proceed with the Lorentzian because it fits naturally into the Gaussian exponent, giving:


$$
\begin{align*}
\sigma_{w_{r_i c_j}}^2 &= \frac{\sigma_{0}^2}{1 + \alpha\sum_{k=0}^{i-1}w_{r_k c_j}^2} \\
&= \frac{\sigma_{0}^2}{\phi(w, i-1, c_j)} \quad \text{where} \quad \phi(w, i, c_j)=1 + \alpha\sum_{k=0}^{i}w_{r_k c_j}^2 \\[14pt]
p(w_{r_i c_j}|w_{r_0 c_j}, w_{r_1 c_j}, \ldots, w_{r_{i-1} c_j}) &= \mathcal{N}(0, \sigma_{w_{r_i c_j}}^2) \\
&= \frac{1}{\sqrt{2\pi\sigma_{w_{r_i c_j}}^2}}\exp\left(\frac{-w_{r_i c_j}^2}{2\sigma_{w_{r_i c_j}}^2}\right) \\
&= \frac{\sqrt{\phi(w,i-1,c_j)}}{\sqrt{2\pi}\sigma_0}\exp\left(\frac{-w_{r_i c_j}^2\phi(w,i-1,c_j)}{2\sigma_0^2}\right) \\[14pt]
\end{align*}
$$

Substituting in the main expression:
$$
\begin{align*}
p(\vec{w}_{c_j}) &= p(w_{r_0 c_j})p(w_{r_1 c_j}|w_{r_0 c_j})p(w_{r_2 c_j}|w_{r_0 c_j}, w_{r_1 c_j})\cdots p(w_{r_{K-1} c_j}|w_{r_0 c_j}, \ldots, w_{r_{K-2} c_j}) \\[6pt]
&= \prod_{i=0}^{K-1} \frac{\sqrt{\phi(w,i-1,c_j)}}{\sqrt{2\pi}\sigma_0}\exp\left(\frac{-w_{r_i c_j}^2\phi(w,i-1,c_j)}{2\sigma_0^2}\right) \\
p(W) &= \prod_{j=0}^{D-1} p(\vec{w}_{c_j}) \\
&= \prod_{j=0}^{D-1} \prod_{i=0}^{K-1} \frac{\sqrt{\phi(w,i-1,c_j)}}{\sqrt{2\pi}\sigma_0}\exp\left(\frac{-w_{r_i c_j}^2\phi(w,i-1,c_j)}{2\sigma_0^2}\right)
\end{align*}
$$

::: {.callout-important}
$p(\vec{w}_{c_j})$ is asymmetric in the row ordering: earlier rows are favored to take non-zero values. In practice this does not appear to affect the learned representations — averaging the loss over all cyclic permutations of the row ordering yields identical results.
:::

## The full distribution

$$
\begin{gather*}
p(W\vec{s}|\vec{x}) \propto \mathcal{N}(\vec{x}|W\vec{s}, \sigma_{\epsilon}^2) \mathcal{N}(\vec{s}|0, \sigma_s^2) \prod_{j=0}^{D-1} \prod_{i=0}^{K-1}\mathcal{N}(w_{r_i c_j}|\sigma_{w_{r_i c_j}}^2) \\
\sigma_{w_{r_i c_j}}^2 = \frac{\sigma_0^2}{\phi(w, i-1, c_j)} \quad \text{where} \quad \phi(w, i, c_j)=1 + \alpha\sum_{k=0}^{i}w_{r_k c_j}^2
\end{gather*}
$$

# Loss function

We take the negative log of the probability distribution to get the loss function.  

$$
\begin{align*}
-\ln(p(W)) &= -\ln\left(\prod_{j=0}^{D-1} \prod_{i=0}^{K-1}\mathcal{N}(w_{r_i c_j}|\sigma_{w_{r_i c_j}}^2)\right)  \\
&= \frac{1}{2} \sum_{j=0}^{D-1} \sum_{i=0}^{K-1} \left[\ln(2\pi) + \ln(\sigma_{w_{r_i c_j}}^2) + \frac{w_{r_i c_j}^2}{\sigma_{w_{r_i c_j}}^2} \right] \\
&= \frac{1}{2} \sum_{j=0}^{D-1} \sum_{i=0}^{K-1} \left[ \ln(2\pi) + \ln(\sigma_0^2) - \ln(\phi(w,i-1,c_j)) + \frac{w_{r_i c_j}^2\phi(w,i-1,c_j)}{\sigma_0^2} \right] \\
\end{align*}
$$

Note that $\ln(p(W))$ is written in the form of each number $w_{r_i c_j}$, we therefore also write $p(\vec{s})$ and $p(\vec{x}|W,\vec{s})$ in the form of each number in the vector. Not doing this can result in handling the dimensions incorrectly later.  

$$
\begin{align*}
-\ln (p(\vec{x}|W,\vec{s})) &= \frac{1}{2} \sum_{j=0}^{D-1} \left[\ln(2\pi) + \ln(\sigma_{\epsilon}^2) + \frac{(x_j-W_{c_j}s_j)^2}{\sigma_{\epsilon}^2} \right] \\
-\ln (p(\vec{s})) &= \frac{1}{2} \sum_{j=0}^{D-1} \left[ \ln(2\pi) + \ln(\sigma_s^2) + \frac{s_j^2}{\sigma_s^2} \right] \\
\end{align*}
$$

Removing the terms containing only constants (including the hyperparameters $\sigma_s, \sigma_0, \sigma_{\epsilon}$)

$$
\begin{align*}
-\ln(p(W\vec{s}|\vec{x})) &\propto -\ln(p(\vec{x}|W\vec{s})) - \ln(P(W|\vec{s})) - \ln(p(\vec{s})) \\
&\propto \sum_{j=0}^{D-1} \left[
    \frac{(\vec{x}-W\vec{s})^2}{\sigma_{\epsilon}^2}
    + \frac{s^2}{\sigma_s^2}
    + \sum_{i=0}^{K-1}\left(-\ln(\phi(w,i-1,c_j)) + \frac{w_{r_i c_j}^2\phi(w,i-1,c_j)}{\sigma_0^2} \right)
    \right] \\
\end{align*} 
$$

Seeing the terms inside the outer sum, we can now deduce that the loss function is saying:

- Loss of each term inside $X$ is balanced with each term inside $S$, which is in turn balanced with each **column** inside $W$ (since we do a summation inside).  

We can take out the outer sum for simplicity.  

$$
-\ln(p(W\vec{s}|\vec{x})) 
\propto 
    \frac{(\vec{x}-W\vec{s})^2}{\sigma_{\epsilon}^2}
    + \frac{s^2}{\sigma_s^2}
    + \sum_{i=0}^{K-1}\left(-\ln(\phi(w,i-1,c_j)) + \frac{w_{r_i c_j}^2\phi(w,i-1,c_j)}{\sigma_0^2} \right)
$$

This will translate to code now as:
- For each sample, we take the mean of individual MSE components, mean of the individual $s_{ij}$ components, and the mean of the disjoint loss across columns (the disjoint loss is summed inside the column).  


Some simple substitutions, useful later. Most of the loss components are pretty standard.  
$$
\begin{align*}
\text{MSE}(X - WS) &=\frac{(\vec{x}-W\vec{s})^2}{\sigma_{\epsilon}^2}  \\
\text{L2}(S) &= \frac{s^2}{\sigma_s^2} \\
\text{DisjointLoss}(W) &= \sum_{i=0}^{K-1}\left[-\ln(\phi(w,i-1,c_j)) + \frac{w_{r_i c_j}^2\phi(w,i-1,c_j)}{\sigma_0^2} \right] \\
\end{align*}
$$

## Hyperparameters

We have 4 hyperparameters: $\sigma_{\epsilon}, \sigma_0, \sigma_s, \alpha$.  
These hyperparamaters are quite sensitive, and it is useful to understand what they mean, giving us good starting points.  

$\sigma_0$ and $\alpha$ are very tightly connected, we'll create a simple formula for an $\alpha$ value which works generally.  

### $\sigma_0$ and $\alpha$

$\sigma_0$ is the standard deviation of the Gaussian from which $w_{rc}$ is sampled when no preceding entry in the column is non-zero. $\alpha$ controls how rapidly this standard deviation shrinks for subsequent entries:

$$\sigma_{w_{rc}}^2 = \frac{\sigma_0^2}{1 + \alpha\sum_{i=0}^{r-1}w_{ic}^2} = \frac{\sigma_0^2}{1 + \alpha x^2}$$

The plots below show this function for $\sigma_0=1$ and $\sigma_0=0.1$ across the same set of $\alpha$ values. For smaller $\sigma_0$, the same $\alpha$ produces a slower collapse. We need a narrower curve. It seems $\alpha$ and $\sigma_0$ are not independent.   



In [ ]:
# | code-fold: true
# | code-summary: code helper plotting functions

## helper plotting functions
import numpy as np
import matplotlib.pyplot as plt

def _single_plot(ax, sigma_0, alphas, x_limit=None):
    if x_limit is not None:
        lo, hi = x_limit
        x = np.concatenate([np.linspace(lo, 0, 250), np.linspace(0, hi, 251)[1:]])

    for alpha in alphas:
        sigma = np.sqrt(sigma_0**2 / (1 + (alpha * (x**2))))
        ax.plot(x, sigma, label=f"alpha={alpha}")

    ax.set_title(f"sigma_0^2={sigma_0**2:.2e}")
    ax.set_xlabel("x")
    ax.set_ylabel("sigma(x)")
    if x_limit is not None:
        ax.set_xlim(*x_limit)
        ax.set_ylim(0, x_limit[1])
    ax.legend()
    ax.grid(True, alpha=0.3)


def plot_lorentz_for_sigma_0_and_alpha_compare(
    examples
):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle("plot of sigma_w_rc^2 wrt sigma_0^2 and alpha")
    _single_plot(axes[0], examples[0]["sigma_0"], examples[0]["alphas"], examples[0]["x_limits"])
    _single_plot(axes[1], examples[1]["sigma_0"], examples[1]["alphas"], examples[1]["x_limits"])
    plt.tight_layout()
    plt.show()

In [ ]:
# | code-fold: true

examples = [
    {
        "sigma_0": 1.0,
        "alphas": [0.5, 5.0, 1000, 10_000],
        "x_limits": [-1.0, 1.0],
    },
    {
        "sigma_0": 0.1,
        "alphas": [0.5, 5.0, 1000, 10_000],
        "x_limits": [-0.1, 0.1],
    },
]

plot_lorentz_for_sigma_0_and_alpha_compare(examples)

The plots below use $\frac{\alpha}{\sigma_0^2}$ as the effective hyperparameter across $\sigma_0 \in [1, 10^{-1}, 10^{-3}, 10^{-4}]$; the curves are identical across all values, confirming this scaling.

::: {.callout-important}
$\alpha$ should be set inversely proportional to $\sigma_0^2$. Since $w_{rc} \sim \mathcal{N}(0, \sigma_0^2)$, we have $x^2 \in [0, 3\sigma_0^2]$ with high probability, making $\alpha = \frac{c}{\sigma_0^2}$ a natural choice. Empirically, $c \in [1000, 10000]$ works well. This keeps $\alpha x^2$ stable across different $\sigma_0$ values.
:::

<!-- We need a thinner curve for $\sigma_0=0.01$.  

::: {.callout-important}
$\alpha$ is not a simple scalar. It should be inversely proportional to $x^2$. Since $x$ itself is sampled from $\mathcal{N}(0, \sigma_0^2)$, $x^2$ is in the range $[-3\sigma_0^2, 3\sigma_0^2]$, giving us using $\alpha = \frac{constant}{\sigma_0^2}$ a good heuristic.  
A useful empirical value is the range $constant \in [1000, 10,000]$.  
What this basically does is keep $\alpha x^2$ predictable and stable across different $\sigma_0$ values.  
:::

The next plots use $\frac{\alpha}{\sigma_0^2}$ as the effective $\alpha$ value. Using $\sigma_0 \in [1, 1e-1, 1e-3, 1e-4]$.   
Notice that with this small change, we get identical plots for all $\sigma_0$ values.   -->

In [ ]:
# | code-fold: true
def _d(a, s):
    return a / (s*s)

examples = [
    {
        "sigma_0": 1.0,
        "alphas": [_d(a, 1.0) for a in [0.5 , 5.0, 1000, 10_000]],
        "x_limits": [-1.0, 1.0],
    },
    {
        "sigma_0": 0.1,
        "alphas": [_d(a, 0.1) for a in [0.5 , 5.0, 1000, 10_000]],
        "x_limits": [-0.1, 0.1],
    },
]

plot_lorentz_for_sigma_0_and_alpha_compare(examples)


examples = [
    {
        "sigma_0": 1e-3,
        "alphas": [_d(a, 1e-3) for a in [0.5 , 5.0, 1000, 10_000]],
        "x_limits": [-1e-3, 1e-3],
    },
    {
        "sigma_0": 1e-4,
        "alphas": [_d(a, 1e-4) for a in [0.5 , 5.0, 1000, 10_000]],
        "x_limits": [-1e-4, 1e-4],
    },
]

plot_lorentz_for_sigma_0_and_alpha_compare(examples)


examples = [
    {
        "sigma_0": 10.0,
        "alphas": [_d(a, 10.0) for a in [0.5 , 5.0, 1000, 10_000]],
        "x_limits": [-10.0, 10.0],
    },
    {
        "sigma_0": 10000.,
        "alphas": [_d(a, 10000.) for a in [0.5 , 5.0, 1000, 10_000]],
        "x_limits": [-10000., 10000.],
    },
]

plot_lorentz_for_sigma_0_and_alpha_compare(examples)

### ${\sigma_{\epsilon}}$

The model assumes the error is distributed under $\mathcal{N}(0, \sigma_{\epsilon})$. We want a good estimate of this.  
The easiest way is to simply minimise **only** reconstruction loss ($MSE$). The code uses a very basic autoencoder (covered in the next section). We simply train a baseline autoencoder to do this. And simply use: 
$$\sigma_{\epsilon} = stddev((X - Reconstruction_{baseline}))$$

::: {.callout-caution}
On pure data, without any noise, this can give $0$, which is a bit extreme. I have not explored this case though. We might need to set some empirical minimum value based on the variance of the input dataset
:::

You would want $\sigma_0$ now to be bigger than $\sigma_{\epsilon}$, but still be reasonably small.  
Basic heuristic $\sigma_0 = 5\sigma_{\epsilon}$ works reasonably well empirically.  

### $\sigma_s$

We want $\vec{s}$ to be less constrained than $W$, I've simply set the value to $5\sigma_0$ and it has empirically worked in practice until now. There is no watertight relation though.  

### summary

$$
\begin{align*}
\sigma_{\epsilon} &= stddev((X - Reconstruction_{baseline})) \\
\sigma_0 &= 5\sigma_{\epsilon} \\
\alpha &= \frac{5000}{\sigma_0^2} \\
\sigma_s &= 5\sigma_0 \\
\end{align*}
$$

# An autoencoder with our loss function

The architecture follows a standard autoencoder design, similar to a Sparse Autoencoder (SAE) — a single-layer encoder-decoder network trained to reconstruct its input through a bottleneck.


Given input $\vec{x}$, the encoder projects it to a latent representation $\vec{s} = \vec{x}E$, and the decoder reconstructs it as $\hat{\vec{x}} = \vec{s}W$. The only departure from a vanilla SAE is the loss function.

To map this to code:

- The intermediate activations from the encoder are $\vec{s}$
- The `nn.Linear` decoder layer is $W$
- The L2 penalty is on the encoder weights $E$, not on $\vec{s}$ directly


```{mermaid}
flowchart LR
    X["X (input)"] --> ENC["nn.Linear\n[encoder]"]
    ENC --> ACT["vector: S \n[intermediate activation]"]
    ACT --> DEC["nn.Linear W\n[decoder]"]
    DEC --> RECON["X (reconstruction)"]

    L2["L2(S)"] -. regularizer .-> ENC
    DS["DisjointLoss(W)"] -. regularizer .-> DEC
    MSE["MSE(X-WS)"] -. objective .-> RECON
```


::: {.callout-note}
A more classical solution like how dictionary learning is implemented in `sklearn` with coordinate descent might work too, I've not tested it out.  
:::

## Why L2 on encoder

- In practice, $\sigma_0$ must be kept very small — this tightens the constraint on $W$. Without it, gradient descent ignores the penalty on $W$ entirely and just minimises MSE.

- $\vec{s}$ must be free to grow. The coefficients will generally need to be much larger in magnitude than the basis vectors.
  - Directly regularising $\vec{s}$ is counterproductive: the penalty would shrink its magnitude, making reconstruction impossible — you can't satisfy $\vec{x} = W\vec{s}$ when both $W$ and $\vec{s}$ are vanishingly small.
  - The goal isn't small $\vec{s}$, it's *well-behaved* $\vec{s}$. We want to penalise gradient descent for exploiting $\vec{s}$ in degenerate ways, not for letting it scale.
  - Concretely, we want $\vec{s}$ to be uniform — low variance across its components, not low magnitude.

- In the autoencoder setting, $\vec{s} = \vec{x}E$ where $E$ are the encoder weights. Modelling these as Gaussian is natural: the product of a Gaussian matrix with a fixed input is itself Gaussian, so the distributional assumption carries through cleanly.

## Code for the model

This section is going to be code-heavy. I've not collapsed the sections containing the definition of the autoencoder and it's training schedule, those are important parts.   
I'm also going to run the model on the generated synthetic data, which gives similar results to the ones shown in the first section where I compare this with different algorithms.  

In [ ]:
import torch
from torch import nn
from torch import optim


def _get_weights_loss_on_decoder(model, alpha, sigma_0, weights_algo):
    if weights_algo == "cycled":
        weight_loss = model.weights_loss_cycled(alpha, sigma_0, model.decoder.weight)
    else:
        comp1, comp2 = model.weights_loss(alpha, sigma_0, model.decoder.weight)
        weight_loss = (comp1 + comp2).sum()
    return weight_loss


class Autoencoder(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, n_components, bias=True),
        )
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        # coefficients is the vector S
        # self.decoder = W
        # x = X
        # recon = Reconstruction
        coefficients = self.encoder(x)
        recon = self.decoder(coefficients)
        return recon, coefficients

    def recon_loss(self, x, recons, sigma_x):
        """Reconstruction loss. MSE"""
        return self.gauss_loss(x, recons) / (sigma_x * sigma_x)

    def coefficients_loss(self, sigma_s):
        """L2 loss on encoder"""
        return self.gauss_loss(self.encoder[0].weight, 0) / (sigma_s * sigma_s)

    def gauss_loss(self, x, mean):
        loss = (x - mean) ** 2
        return torch.sum(loss, 1).mean()

    def weights_loss_cycled(self, alpha, sigma_0, W):
        """
        This is similar to weight_loss function
        We basically run `weight_loss` starting from every dimension c, and then average them out.
        Useful for testing if there is a bias in the main `weights_loss` function
        """
        K = W.shape[1]
        shift_losses = []
        for s in range(K):
            comp1, comp2 = self.weights_loss(alpha, sigma_0, torch.roll(W, -s, dims=1))
            shift_losses.append((comp1 + comp2).sum())
        return torch.mean(torch.stack(shift_losses))

    def weights_loss(self, alpha, sigma_0, W):
        """Vectorized version the Weight loss"""
        W_sq = W**2  # (C, K)
        cumsum = torch.cumsum(W_sq, dim=1)  # (C, K), cumsum[c,k] = sum W[c,0..k]^2
        phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
        phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
        comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
        comp2 = -torch.log(phi)
        return comp1, comp2


def train(
    X,
    n_components,
    alpha=5000,
    sigma_eps=0.1,
    sigma_s=1,
    sigma_0=1,
    lr=1e-3,
    epochs=2000,
    batch_size=256,
    weights_algo="cycle",
    verbose=True,
):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    sigma_eps: std of noise in the data after modelling the data as a W@S
    sigma_0: std of W, useful to keep very near 0
    sigma_s: sigma for the gaussian distribution for sampling the encoder weights. It indirectly restricts its outputs (the coefficients) to be gaussian
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = Autoencoder(input_dim, n_components)

    # svd initialisation helps, but is not very necessary if the hyperparameters are correctly tuned
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)

            # the main loss computation code, most relevant piece
            recon_loss = model.recon_loss(batch, recon, sigma_eps)
            coefficients_loss = model.coefficients_loss(sigma_s)
            weight_loss = _get_weights_loss_on_decoder(
                model, alpha, sigma_0, weights_algo
            )
            loss = recon_loss + weight_loss + coefficients_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if verbose and epoch % 200 == 0:
            print(
                f"epoch {epoch:4d} | recon_loss {recon_loss:.4f} weight_loss {weight_loss.sum():.4f} coefficients_loss {coefficients_loss:.4f}"
            )

    with torch.no_grad():
        recon, codes = model(X_t)
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),
        recon.numpy(),
    )


def train_baseline(
    X, n_components, lr=1e-3, epochs=2000, batch_size=256, sigma_x=1, verbose=True
):
    """Train the autoencoder with just reconstruction loss, to find an arbitrary linear model which fits the data

    The main training code requires sigma_eps
    the standard deviation of expected gaussian noise
    when the curve is fitted using Y=WX
    We can generally do a simple sweep of hyperparams
    or use simple heuristics
    If the data is linearly "fittable",
    then we get a good starting point
    using this function. 
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = Autoencoder(input_dim, n_components)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            loss = recon_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if verbose and epoch % 200 == 0:
            print(f"finetune epoch {epoch:4d} | recon_loss {recon_loss:.4f}")

    with torch.no_grad():
        recon, codes = model(X_t)

    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

## Testing on synthetic data

In [ ]:
#| code-fold: true
#| code-summary: Code for Synthetic data generation script
N_COMPS = 3
N_DIM = 9
X, W_true, codes_true, dim_partition = generate_synthetic_patches(
    N_DIM, N_COMPS, seed=10
)


ws_to_show = [w.reshape(3, 3) for w in W_true]
S(
    ws_to_show,
    (5, 2),
    ncols=3,
    mode=MODE,
    suptitle="ground truth basis vectors \nA vector is of size 1x9, but is shown as 3x3 for easier visibility",
)
plt.show()

S(
    [X[0].reshape(3, 3)] + ws_to_show,
    (6, 2),
    4,
    suptitle="First input, with its basis components, the coefficient of each component is it's title",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[0][0]:.3f}",
        f"{codes_true[0][1]:.3f}",
        f"{codes_true[0][2]:.3f}",
    ],
)
plt.show()

In [ ]:
#| code-fold: true
#| code-summary: Code for model training script

# we first train a baseline model without any of the regularisations on this dataset
# this would help us find the "best fit"
# this gives us sigma_eps (the std dev of the noise) when the data is fitted using WS
model, codes, components, recon = train_baseline(X, 3, epochs=600, verbose=False)

std_eps = (X - recon).std()
# sigma_0 should be tight, empirically these values are working fine
sigma_0 = std_eps * 5
sigma_s = sigma_0 * 10

# alpha and sigma_0 are tied together. more on this relationship later
alpha = 5000 * (1 / (sigma_0 * sigma_0))
print(
    f"using parameters: sigma_0={sigma_0:.4f} sigma_s={sigma_s:.4f} sigma_eps={std_eps:.4f} alpha={alpha:.4f}"
)

model, codes, components, recon = train(
    X,
    3,
    alpha=alpha,
    sigma_eps=std_eps,
    sigma_0=sigma_0,
    sigma_s=sigma_s,
    weights_algo="no-cycle",
    verbose=False,
)
show_closest_component_of_W_for_each_component(components, W_true)

Lets look at some of the reconstructions and their components

In [ ]:
# | code-fold: true
ws_to_show = [w.reshape(3, 3) for w in components]


i = 0
S(
    [X[i].reshape(3, 3)] + ws_to_show,
    (6, 2),
    4,
    suptitle=f"Reconstruction {i}",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[i][0]:.3f}",
        f"{codes_true[i][1]:.3f}",
        f"{codes_true[i][2]:.3f}",
    ],
)
plt.show()

i = 1
S(
    [X[i].reshape(3, 3)] + ws_to_show,
    (6, 2),
    4,
    suptitle=f"Reconstruction {i}",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[i][0]:.3f}",
        f"{codes_true[i][1]:.3f}",
        f"{codes_true[i][2]:.3f}",
    ],
)
plt.show()

i = 1
S(
    [X[i].reshape(3, 3)] + ws_to_show,
    (6, 2),
    4,
    suptitle=f"Reconstruction {i}",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[i][0]:.3f}",
        f"{codes_true[i][1]:.3f}",
        f"{codes_true[i][2]:.3f}",
    ],
)
plt.show()